# Этап 1. Загрузка и профилирование данных

Загрузка `all_regions.csv` (~2 ГБ) через DuckDB, задание типов и сохрание данные в Parquet со сжатием Zstandard. Проверка структуры, пропусков, диапазонов значений и повторных наблюдений объявлений.


## Исходные данные

| Колонка | Тип | Описание |
| --- | --- | --- |
| `brand`, `name`, `bodyType`, `color`, `fuelType`, `transmission`, `vehicleConfiguration`, `engineName`, `engineDisplacement`, `location`, `link`, `description` | `VARCHAR` | Текст |
| `year` | `SMALLINT` | Год выпуска |
| `mileage` | `INTEGER` | Пробег, км |
| `power` | `SMALLINT` | Мощность, л.с. |
| `price` | `BIGINT` | Цена, руб. |
| `date` | `TIMESTAMP` | Дата публикации объявления |
| `parse_date` | `TIMESTAMP` | Дата сбора данных |

## Подготовка и пути

Ноутбук можно запускать из корня проекта или из папки `notebooks`. Исходный CSV должен находиться в `data/raw/all_regions.csv`.

`REBUILD_PARQUET = False` позволяет использовать уже созданный Parquet. После замены CSV или изменения правил приведения типов нужно установить `True`, выполнить ноутбук и вернуть флаг в `False` для обычной работы.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 90

working_dir = Path.cwd().resolve()

if (working_dir / "notebooks").is_dir() and (working_dir / "data").is_dir():
    BASE_DIR = working_dir
elif working_dir.name == "notebooks" and (working_dir.parent / "data").is_dir():
    BASE_DIR = working_dir.parent
else:
    raise FileNotFoundError("Запустите ноутбук из корня проекта или папки notebooks.")

RAW_CSV = BASE_DIR / "data" / "raw" / "all_regions.csv"
OUTPUT_PARQUET = BASE_DIR / "data" / "processed" / "raw_typed.parquet"
REBUILD_PARQUET = False

RAW_CSV, OUTPUT_PARQUET

(WindowsPath('E:/Dev/car-market-analysis/data/raw/all_regions.csv'),
 WindowsPath('E:/Dev/car-market-analysis/data/processed/raw_typed.parquet'))

## Загрузка и сохранение в Parquet

CSV сначала читается как текст (`all_varchar = true`), затем числовые поля и даты приводятся к типам из таблицы выше. Пустые поля CSV читаются как `NULL`. Папка для результата создаётся автоматически.

In [2]:
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

if REBUILD_PARQUET or not OUTPUT_PARQUET.exists():
    if not RAW_CSV.is_file():
        raise FileNotFoundError(f"Не найден исходный CSV: {RAW_CSV}")

    OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    con.execute(
        """
        CREATE OR REPLACE TABLE cars_csv AS
        SELECT
            brand,
            name,
            bodyType,
            color,
            fuelType,
            CAST(year AS SMALLINT) AS year,
            CAST(mileage AS INTEGER) AS mileage,
            transmission,
            CAST(power AS SMALLINT) AS power,
            CAST(price AS BIGINT) AS price,
            vehicleConfiguration,
            engineName,
            engineDisplacement,
            CAST(date AS TIMESTAMP) AS date,
            location,
            link,
            description,
            CAST(parse_date AS TIMESTAMP) AS parse_date
        FROM read_csv(
            ?,
            header = true,
            delim = ',',
            quote = '"',
            escape = '"',
            encoding = 'utf-8',
            all_varchar = true
        )
        """,
        [str(RAW_CSV)],
    )

    output_sql_path = OUTPUT_PARQUET.as_posix().replace("'", "''")
    con.execute(
        f"COPY cars_csv TO '{output_sql_path}' (FORMAT parquet, COMPRESSION zstd)"
    )

    con.execute("DROP TABLE cars_csv")
    print("Parquet создан из исходного CSV.")
else:
    print("Используется существующий Parquet.")

con.read_parquet(str(OUTPUT_PARQUET)).create_view("cars_raw", replace=True)

con.execute("SELECT count(*) AS rows_loaded FROM cars_raw").df()

Используется существующий Parquet.


,rows_loaded
0,1294757


In [3]:
schema = con.execute("DESCRIBE cars_raw").df()
schema

,column_name,column_type,null,key,default,extra
0,brand,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,bodyType,VARCHAR,YES,None,None,None
3,color,VARCHAR,YES,None,None,None
4,fuelType,VARCHAR,YES,None,None,None
5,year,SMALLINT,YES,None,None,None
6,mileage,INTEGER,YES,None,None,None
7,transmission,VARCHAR,YES,None,None,None
8,power,SMALLINT,YES,None,None,None
9,price,BIGINT,YES,None,None,None


In [4]:
file_sizes = pd.DataFrame(
    {
        "file": [RAW_CSV.name, OUTPUT_PARQUET.name],
        "size_mb": [
            RAW_CSV.stat().st_size / 1024**2 if RAW_CSV.exists() else None,
            OUTPUT_PARQUET.stat().st_size / 1024**2,
        ],
    }
)
file_sizes

,file,size_mb
0,all_regions.csv,"2,043.31"
1,raw_typed.parquet,224.54


## Контроль загрузки

Сравнивнение количества строк и пропусков `year` в исходном CSV и Parquet.

In [5]:
parquet_check = con.execute(
    """
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (WHERE year IS NULL) AS year_nulls
    FROM cars_raw
    """
).df()

total_rows = int(parquet_check.loc[0, "rows_count"])
if total_rows == 0:
    raise ValueError("Parquet не содержит строк для профилирования.")

if RAW_CSV.is_file():
    raw_check = con.execute(
        """
        SELECT
            count(*) AS rows_count,
            count(*) FILTER (WHERE year IS NULL) AS year_nulls
        FROM read_csv(
            ?, header = true, delim = ',', quote = '"', escape = '"',
            encoding = 'utf-8', all_varchar = true
        )
        """,
        [str(RAW_CSV)],
    ).df()

    if not raw_check.equals(parquet_check):
        raise ValueError("Количество строк или пропусков year в CSV и Parquet различается.")

    display(pd.concat({"CSV": raw_check, "Parquet": parquet_check}, names=["source"]))
else:
    print("Исходный CSV отсутствует: сверка пропущена.")
    display(parquet_check)

,,rows_count,year_nulls
source,,,
CSV,0,1294757,570113
Parquet,0,1294757,570113


## Пропуски и пустые строки

Для каждой колонки: количество и доля `NULL`. Для текстовых полей: проверка непустых с точки зрения SQL значений, состоящих только из пробелов. В числовых и временных колонках показатель `blank_count` равен нулю, поскольку они уже типизированы.

Прогресс-бар в столбце `null_pct` показывает долю пропусков по шкале от 0 до 100%. Все доли рассчитаны от общего количества строк.

In [6]:
non_null_counts = con.execute("SELECT count(COLUMNS(*)) FROM cars_raw").df().iloc[0]
missing = (total_rows - non_null_counts).to_frame("null_count")
missing["null_pct"] = 100 * missing["null_count"] / total_rows

text_columns = schema.loc[schema["column_type"] == "VARCHAR", "column_name"].tolist()
blank_expressions = [
    f'count(*) FILTER (WHERE "{column}" IS NOT NULL AND length(trim("{column}")) = 0) AS "{column}"'
    for column in text_columns
]
blank_counts = con.execute("SELECT " + ", ".join(blank_expressions) + " FROM cars_raw").df().iloc[0]
missing["blank_count"] = blank_counts.reindex(missing.index, fill_value=0).astype("int64")
missing = missing.sort_values("null_count", ascending=False)
missing.index.name = "column_name"

missing.style.format({"null_pct": "{:.2f}"}).bar(
    subset=["null_pct"], color="#93c5fd", vmin=0, vmax=100
)

,null_count,null_pct,blank_count
column_name,,,
engineDisplacement,577132,44.57,0
engineName,573781,44.32,0
year,570113,44.03,0
vehicleConfiguration,570110,44.03,0
mileage,522958,40.39,0
description,40352,3.12,0
color,37728,2.91,0
power,21404,1.65,0
transmission,5194,0.40,0


## Диапазоны числовых полей и дат

Для числовых полей: количество известных значений, минимум, максимум, медиану и квантили.

In [7]:
numeric_columns = ["year", "mileage", "power", "price"]
numeric_queries = [
    f"""
    SELECT
        '{column}' AS column_name,
        count({column}) AS non_null_count,
        min({column}) AS min_value,
        quantile_cont({column}, 0.01) AS p01,
        quantile_cont({column}, 0.25) AS p25,
        quantile_cont({column}, 0.50) AS median,
        quantile_cont({column}, 0.75) AS p75,
        quantile_cont({column}, 0.99) AS p99,
        max({column}) AS max_value
    FROM cars_raw
    """
    for column in numeric_columns
]
numeric_profile = con.execute(" UNION ALL ".join(numeric_queries)).df().set_index("column_name")
numeric_profile

,non_null_count,min_value,p01,p25,median,p75,p99,max_value
column_name,,,,,,,,
year,724644,1936,"1,982.00","2,003.00","2,011.00","2,017.00","2,023.00",2023
mileage,771799,1000,"1,000.00","82,000.00","144,000.00","211,000.00","441,000.00",1000000
power,1273353,1,58.00,98.00,128.00,163.00,388.00,1000
price,1294757,270,"59,000.00","425,000.00","870,000.00","1,765,000.00","9,000,000.00",150000000


In [8]:
date_profile = con.execute(
    """
    SELECT 'date' AS column_name, count(date) AS non_null_count,
           min(date) AS min_value, max(date) AS max_value
    FROM cars_raw
    UNION ALL
    SELECT 'parse_date', count(parse_date), min(parse_date), max(parse_date)
    FROM cars_raw
    """
).df().set_index("column_name")

reference_year = int(date_profile.loc["parse_date", "max_value"].year)
date_profile

,non_null_count,min_value,max_value
column_name,,,
date,1294757,2023-03-27 00:00:00,2023-06-21 00:00:00
parse_date,1294757,2023-05-01 21:00:00,2023-06-21 22:00:00


## Предварительные признаки проблем качества

Проверка обязательных для основных расчётов полей, неположительную цену, отрицательный пробег и согласованность дат. Дополнительно отмечаем значения года вне диапазона **1960 — год последнего сбора данных** и мощности вне **20–1000 л.с.**, предложенных в плане проекта.

In [9]:
quality_rules = {
    "Пропуск марки, модели, цены, года или ссылки": """
        brand IS NULL OR trim(brand) = ''
        OR name IS NULL OR trim(name) = ''
        OR price IS NULL OR year IS NULL
        OR link IS NULL OR trim(link) = ''
    """,
    "Неположительная цена": "price <= 0",
    f"Год вне предварительного диапазона 1960–{reference_year}": f"year < 1960 OR year > {reference_year}",
    "Отрицательный пробег": "mileage < 0",
    "Мощность вне предварительного диапазона 20–1000 л.с.": "power < 20 OR power > 1000",
    "Дата публикации позже даты сбора": "date > parse_date",
}

quality_expressions = [
    f"count(*) FILTER (WHERE {condition}) AS rule_{index}"
    for index, condition in enumerate(quality_rules.values())
]
quality_counts = con.execute("SELECT " + ", ".join(quality_expressions) + " FROM cars_raw").fetchone()
quality_profile = pd.DataFrame(
    {"check": list(quality_rules), "rows_count": quality_counts}
).set_index("check")
quality_profile["rows_pct"] = 100 * quality_profile["rows_count"] / total_rows

combined_condition = " OR ".join(f"({condition})" for condition in quality_rules.values())
flagged_rows = con.execute(
    f"SELECT count(*) FROM cars_raw WHERE {combined_condition}"
).fetchone()[0]
quality_profile.loc["Хотя бы один из перечисленных признаков"] = [
    flagged_rows, 100 * flagged_rows / total_rows
]
quality_profile["rows_count"] = quality_profile["rows_count"].astype("int64")
quality_profile

,rows_count,rows_pct
check,,
"Пропуск марки, модели, цены, года или ссылки",570113,44.03
Неположительная цена,0,0.00
Год вне предварительного диапазона 1960–2023,405,0.03
Отрицательный пробег,0,0.00
Мощность вне предварительного диапазона 20–1000 л.с.,55,0.00
Дата публикации позже даты сбора,3,0.00
Хотя бы один из перечисленных признаков,570521,44.06


## Полные дубли и повторные ссылки

- **Полные дубли** — строки, одинаковые по всем 18 колонкам; подсчет количества копий сверх первой.
- **Повторяющиеся ссылки** — объявления, для которых есть несколько строк, в том числе наблюдения в разные даты.
- **Строки сверх одной на ссылку** — сколько записей пришлось бы убрать при выборе одного наблюдения на каждый непустой `link`.

In [10]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE link_profile AS
    SELECT
        link,
        count(*) AS rows_count,
        count(DISTINCT parse_date) AS parse_dates,
        count(DISTINCT price) AS price_versions,
        count(DISTINCT (year IS NULL, year)) AS year_versions,
        count(DISTINCT (mileage IS NULL, mileage)) AS mileage_versions,
        count(DISTINCT (power IS NULL, power)) AS power_versions,
        count(DISTINCT (brand IS NULL, brand)) AS brand_versions,
        count(DISTINCT (name IS NULL, name)) AS name_versions,
        min(parse_date) AS first_parse_date,
        max(parse_date) AS last_parse_date,
        min(price) AS min_price,
        max(price) AS max_price
    FROM cars_raw
    WHERE link IS NOT NULL AND trim(link) <> ''
    GROUP BY link
    """
)

unique_full_rows = con.execute("SELECT count(*) FROM (SELECT DISTINCT * FROM cars_raw)").fetchone()[0]
full_duplicate_rows = total_rows - unique_full_rows
unique_links, repeated_links, repeated_rows, extra_link_rows = con.execute(
    """
    SELECT
        count(*),
        count(*) FILTER (WHERE rows_count > 1),
        coalesce(sum(rows_count) FILTER (WHERE rows_count > 1), 0),
        coalesce(sum(rows_count - 1), 0)
    FROM link_profile
    """
).fetchone()

duplicates_summary = pd.DataFrame(
    {
        "Показатель": [
            "Строк всего",
            "Уникальных непустых ссылок",
            "Ссылок с несколькими наблюдениями",
            "Строк в группах повторных ссылок",
            "Строк сверх одной на ссылку",
            "Доля строк сверх одной на ссылку",
            "Полных дублей сверх первой копии",
        ],
        "Значение": [
            f"{total_rows:,}", f"{unique_links:,}", f"{repeated_links:,}",
            f"{repeated_rows:,}", f"{extra_link_rows:,}",
            f"{100 * extra_link_rows / total_rows:.2f}%", f"{full_duplicate_rows:,}",
        ],
    }
)
duplicates_summary

,Показатель,Значение
0,Строк всего,"1,294,757"
1,Уникальных непустых ссылок,"950,457"
2,Ссылок с несколькими наблюдениями,"213,669"
3,Строк в группах повторных ссылок,"557,969"
4,Строк сверх одной на ссылку,"344,300"
5,Доля строк сверх одной на ссылку,26.59%
6,Полных дублей сверх первой копии,0


### Чем отличаются наблюдения одного объявления

Для повторяющихся ссылок: проверка различия цены, времени сбора и основных характеристик. Доля рассчитана от количества повторяющихся ссылок.

In [11]:
version_columns = {
    "Разные даты сбора": "parse_dates",
    "Разные цены": "price_versions",
    "Разные значения года, включая пропуски": "year_versions",
    "Разные значения пробега, включая пропуски": "mileage_versions",
    "Разные значения мощности, включая пропуски": "power_versions",
    "Разные марки, включая пропуски": "brand_versions",
    "Разные модели, включая пропуски": "name_versions",
}
version_expressions = [
    f"count(*) FILTER (WHERE {column} > 1) AS metric_{index}"
    for index, column in enumerate(version_columns.values())
]
version_counts = con.execute(
    "SELECT " + ", ".join(version_expressions) + " FROM link_profile WHERE rows_count > 1"
).fetchone()
duplicate_differences = pd.DataFrame(
    {"difference": list(version_columns), "links_count": version_counts}
).set_index("difference")
duplicate_differences["repeated_links_pct"] = (
    100 * duplicate_differences["links_count"] / repeated_links if repeated_links else 0
)
duplicate_differences

,links_count,repeated_links_pct
difference,,
Разные даты сбора,213669,100.00
Разные цены,213669,100.00
"Разные значения года, включая пропуски",5306,2.48
"Разные значения пробега, включая пропуски",84027,39.33
"Разные значения мощности, включая пропуски",1058,0.50
"Разные марки, включая пропуски",0,0.00
"Разные модели, включая пропуски",4,0.00


### Совпадения времени сбора и направление изменения цены

Проверка: встречается ли у одной ссылки несколько цен в одно и то же время сбора. Такие наблюдения неоднозначны для построения истории цен.

Сравнение цены с предыдущим однозначным наблюдением той же ссылки с помощью `lag`. Несколько строк с одинаковыми `link`, `parse_date` и ценой представляют одну точку истории. Точки с несколькими разными ценами исключаются только из этого диагностического расчёта; исходные строки остаются в Parquet.

Количество снижений и повышений относится к переходам между наблюдениями, а количество ссылок — к объявлениям. Одно объявление может содержать и снижение, и повышение. Эти результаты описывают наблюдаемые цены предложений, а не цены сделок.

In [12]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE link_snapshots AS
    SELECT link, parse_date,
           count(*) AS rows_count,
           count(DISTINCT price) AS price_versions,
           min(price) AS price
    FROM cars_raw
    WHERE link IS NOT NULL AND trim(link) <> '' AND parse_date IS NOT NULL
    GROUP BY link, parse_date
    """
)

snapshot_quality = con.execute(
    """
    SELECT
        count(*) AS link_time_groups,
        count(*) FILTER (WHERE rows_count > 1) AS groups_with_multiple_rows,
        count(*) FILTER (WHERE price_versions > 1) AS groups_with_conflicting_prices,
        count(*) FILTER (WHERE price_versions = 0) AS groups_without_price
    FROM link_snapshots
    """
).df()
snapshot_quality

,link_time_groups,groups_with_multiple_rows,groups_with_conflicting_prices,groups_without_price
0,1294757,0,0,0


In [13]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE price_changes AS
    SELECT link, parse_date, price,
           lag(price) OVER (PARTITION BY link ORDER BY parse_date) AS previous_price
    FROM link_snapshots
    WHERE price_versions = 1
    """
)

price_change_summary = con.execute(
    """
    SELECT
        count(*) FILTER (WHERE previous_price IS NOT NULL) AS compared_steps,
        count(*) FILTER (WHERE price < previous_price) AS price_decreases,
        count(*) FILTER (WHERE price > previous_price) AS price_increases,
        count(*) FILTER (WHERE price = previous_price) AS unchanged_prices,
        count(DISTINCT link) FILTER (WHERE price < previous_price) AS links_with_decrease,
        count(DISTINCT link) FILTER (WHERE price > previous_price) AS links_with_increase
    FROM price_changes
    """
).df()
price_change_summary.T.rename(columns={0: "count"})

,count
compared_steps,344300
price_decreases,270408
price_increases,73892
unchanged_prices,0
links_with_decrease,177703
links_with_increase,62562


## Выводы и задачи для очистки

- В CSV и Parquet **1 294 757 строк**; количество строк и пропусков года совпадает. Parquet сохраняет типизированные исходные наблюдения без удаления строк.
- Год выпуска отсутствует в **570 113 строках (44,03%)** уже в исходном CSV. Пробег также часто отсутствует.
- Всего есть **950 457 уникальных ссылок** и **344 300 строк сверх одной на ссылку (26,59%)**. **Полных дублей нет**; совпадений пары `link` и `parse_date` также нет. Повторы представляют наблюдения объявлений в разные даты.
- У **213 669 повторяющихся ссылок** встречаются разные цены и разные даты сбора. Различаются также год у **5 306 ссылок**, пробег у **84 027**, мощность у **1 058** и модель у **4**; для характеристик учтены переходы между пропуском и известным значением.
- Между последовательными наблюдениями найдено **270 408 снижений** и **73 892 повышения** цены; переходов без изменения цены нет. Данные позволяют исследовать наблюдаемые изменения, но частоту изменения цен в этом наборе нельзя автоматически переносить на весь рынок без знания процедуры сбора.
- Цена находится в диапазоне **270–150 000 000 руб.**, год выпуска — **1936–2023**, мощность — **1–1000 л.с.**.
- Хотя бы один предварительный признак проблемы имеет **570 521 строка (44,06%)**. Это объединение условий без двойного счёта, основной вклад дают пропуски года. Среди остальных результатов — **405 строк** с годом до 1960, **55** с мощностью вне предварительного диапазона и **3** с датой публикации позже даты сбора. Итоговую долю нельзя считать окончательной долей ошибочных объявлений: диапазоны ещё требуют проверки, а строки относятся к наблюдениям, а не к уникальным автомобилям.